# Módulo 4 — Introdução à IA no Desenvolvimento

A inteligência artificial generativa está transformando a forma como analistas de dados trabalham. Este notebook explora como usar Claude Code no dia a dia de um profissional do setor elétrico.

## O que vamos ver

1. O que é Claude Code e para que serve
2. Como LLMs auxiliam na análise de dados
3. Fluxo de trabalho: prompt → revisão → execução
4. Demonstração: gerar função de qualidade com Claude
5. Dicas práticas para analistas do setor elétrico

---

> **Importante:** Claude Code é uma ferramenta de **assistência** — você revisa e valida todo o código gerado. Nunca execute código gerado por IA sem entendê-lo.

## 1. O que é Claude Code?

Claude Code é uma ferramenta CLI (Command Line Interface) da Anthropic que integra o modelo de linguagem Claude diretamente ao seu terminal e ambiente de desenvolvimento.

**Diferenças em relação ao ChatGPT/Claude.ai:**

| Característica | Claude.ai (web) | Claude Code (CLI) |
|----------------|-----------------|--------------------|
| Acesso a arquivos | Não (upload manual) | Sim, lê/escreve automaticamente |
| Contexto do projeto | Não | Sim, via CLAUDE.md |
| Executa comandos | Não | Sim (com aprovação) |
| Integração IDE | Não | Sim (VS Code, etc.) |
| Uso típico | Chat, perguntas | Desenvolvimento ativo |

---

**Para instalar:**
```bash
npm install -g @anthropic-ai/claude-code
export ANTHROPIC_API_KEY="sua-chave-aqui"
claude  # inicia a sessão interativa
```

## 2. Como LLMs Auxiliam na Análise de Dados

Grandes Modelos de Linguagem (LLMs) são excelentes para tarefas que envolvem texto e padrões — e código é uma forma de texto com padrões muito regulares.

In [ ]:
# Onde LLMs ajudam vs. onde você ainda precisa de julgamento humano
tarefas = {
    "LLMs são excelentes": [
        "Gerar código boilerplate (conexão, leitura de arquivo, estrutura de função)",
        "Escrever docstrings e documentação técnica",
        "Converter SQL em código Python pandas",
        "Explicar erros e sugerir correções",
        "Refatorar código para melhorar legibilidade",
        "Gerar queries Snowflake a partir de descrição em português",
        "Criar testes unitários para funções existentes",
    ],
    "Ainda precisam de julgamento humano": [
        "Definir as regras de negócio corretas (o que é um 'consumo anormal'?)",
        "Validar se os resultados fazem sentido para o setor",
        "Decisões sobre dados sensíveis (CPF, localização)",
        "Aprovar mudanças em sistemas de produção",
        "Interpretar anomalias que requerem conhecimento de campo",
    ]
}

for categoria, itens in tarefas.items():
    print(f"\n{categoria}:")
    for item in itens:
        print(f"  • {item}")

## 3. Fluxo de Trabalho com Claude Code

O fluxo recomendado é:

```
1. DESCREVER → Explique o problema em linguagem natural
2. REVISAR   → Leia o código gerado com atenção
3. TESTAR    → Execute em dados pequenos primeiro
4. VALIDAR   → Confira os resultados com o negócio
5. ITERAR   → Peça ajustes se necessário
```

Nunca pule a etapa de revisão, especialmente para queries que modificam dados.

## 4. Demonstração: Função Gerada com Auxílio de Claude

O código abaixo foi escrito com o apoio do Claude Code a partir do seguinte prompt:

```
Crie uma função Python que recebe um DataFrame pandas com as colunas
cod_consumidor, nom_consumidor, cpf_cnpj, cod_modalidade_tarifaria,
cod_classe_consumo e vlr_consumo_medio_kwh, e retorna um dict com:
- resumo: contagem total, ativos, inativos
- completude: % de preenchimento por campo obrigatório
- problemas_documento: contagem de CPF/CNPJ ausentes ou inválidos
- outliers_consumo: lista de cod_consumidor com consumo > 3 desvios padrão

Use type hints, docstring em português e trate possíveis erros.
```

In [ ]:
# Código gerado com apoio do Claude Code (revisado e validado pelo analista)
import pandas as pd
import numpy as np
import re
from typing import Dict, Any, List


def analisar_qualidade_cadastro(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Realiza análise de qualidade do cadastro de consumidores.
    
    Args:
        df: DataFrame com colunas do cadastro de UCs.
            Colunas esperadas: cod_consumidor, nom_consumidor, cpf_cnpj,
            cod_modalidade_tarifaria, cod_classe_consumo, vlr_consumo_medio_kwh
    
    Returns:
        Dicionário com análises de:
        - resumo: contagens gerais
        - completude: % de preenchimento por campo
        - problemas_documento: situação do CPF/CNPJ
        - outliers_consumo: registros com consumo anormal
    
    Raises:
        ValueError: se o DataFrame estiver vazio ou faltar colunas obrigatórias
    """
    # Validar entrada
    colunas_obrigatorias = [
        "cod_consumidor", "nom_consumidor", "cpf_cnpj",
        "cod_modalidade_tarifaria", "cod_classe_consumo", "vlr_consumo_medio_kwh"
    ]
    colunas_faltantes = [c for c in colunas_obrigatorias if c not in df.columns]
    if colunas_faltantes:
        raise ValueError(f"Colunas ausentes no DataFrame: {colunas_faltantes}")
    
    if df.empty:
        raise ValueError("DataFrame está vazio")
    
    total = len(df)
    
    # 1. Resumo geral
    ativos = df["flg_ativo"].sum() if "flg_ativo" in df.columns else None
    resumo = {
        "total_registros": total,
        "ativos": int(ativos) if ativos is not None else "coluna flg_ativo ausente",
        "inativos": int(total - ativos) if ativos is not None else "coluna flg_ativo ausente",
    }
    
    # 2. Completude por campo
    completude = {
        col: round(
            (df[col].notna() & (df[col].astype(str).str.strip() != "")).sum() / total * 100, 1
        )
        for col in colunas_obrigatorias
    }
    
    # 3. Problemas no documento (CPF/CNPJ)
    def classificar_doc(doc):
        if pd.isna(doc) or str(doc).strip() == "":
            return "ausente"
        digitos = re.sub(r'\D', '', str(doc))
        if len(digitos) in (11, 14):
            if digitos == digitos[0] * len(digitos):
                return "invalido_sequencia"
            return "valido"
        return "formato_invalido"
    
    tipos_doc = df["cpf_cnpj"].apply(classificar_doc).value_counts().to_dict()
    problemas_documento = {
        "validos": tipos_doc.get("valido", 0),
        "ausentes": tipos_doc.get("ausente", 0),
        "formato_invalido": tipos_doc.get("formato_invalido", 0),
        "invalido_sequencia": tipos_doc.get("invalido_sequencia", 0),
        "pct_problemas": round(
            (1 - tipos_doc.get("valido", 0) / total) * 100, 1
        )
    }
    
    # 4. Outliers de consumo (método: z-score > 3)
    consumo = df["vlr_consumo_medio_kwh"].dropna()
    if len(consumo) > 3:
        media = consumo.mean()
        std = consumo.std()
        z_scores = (df["vlr_consumo_medio_kwh"] - media) / std
        mask_outlier = z_scores.abs() > 3
        outliers = df.loc[mask_outlier, ["cod_consumidor", "nom_consumidor", "vlr_consumo_medio_kwh"]]
        outliers_consumo = outliers.to_dict(orient="records")
    else:
        outliers_consumo = []
    
    return {
        "resumo": resumo,
        "completude": completude,
        "problemas_documento": problemas_documento,
        "outliers_consumo": outliers_consumo
    }


# Testar com os dados do curso
df = pd.read_csv("../modulo2_dados_com_pandas/dados/consumidores_simulado.csv")

resultado = analisar_qualidade_cadastro(df)

print("=== Análise de Qualidade do Cadastro ===")
print("\n--- Resumo ---")
for k, v in resultado["resumo"].items():
    print(f"  {k}: {v}")

print("\n--- Completude (%) ---")
for campo, pct in resultado["completude"].items():
    barra = "█" * int(pct / 5)
    print(f"  {campo:<35}: {pct:>5.1f}% {barra}")

print("\n--- Documentos ---")
for k, v in resultado["problemas_documento"].items():
    print(f"  {k}: {v}")

print(f"\n--- Outliers de Consumo ({len(resultado['outliers_consumo'])}) ---")
for out in resultado["outliers_consumo"]:
    print(f"  UC {out['cod_consumidor']}: {out['nom_consumidor']} — {out['vlr_consumo_medio_kwh']:,.1f} kWh")

## 5. Dicas Práticas para Analistas do Setor Elétrico

In [ ]:
dicas = [
    {
        "titulo": "Sempre inclua o domínio no prompt",
        "ruim": "Crie uma função para validar dados",
        "bom": "Crie uma função Python para validar se a modalidade tarifária (B1, B2, B3, B4, A4) é consistente com a classe de consumo (RESIDENCIAL, COMERCIAL, etc.) conforme PRODIST/ANEEL"
    },
    {
        "titulo": "Peça type hints e docstrings",
        "ruim": "Faça uma função de DEC",
        "bom": "Crie uma função calcular_dec com type hints, docstring em português descrevendo a fórmula ANEEL (DEC = Σ(Di×Ci)/Ct), e exemplos de uso no docstring"
    },
    {
        "titulo": "Especifique as colunas do DataFrame",
        "ruim": "Analise o DataFrame de consumidores",
        "bom": "Analise o DataFrame df com colunas: cod_consumidor (int), cpf_cnpj (str), cod_classe_consumo (str), vlr_consumo_medio_kwh (float). Calcule..."
    },
    {
        "titulo": "Peça tratamento de erros",
        "ruim": "Conecte ao Snowflake",
        "bom": "Crie uma classe SnowflakeConnector com context manager, tratamento de ConnectionError, e log de erros. Use variáveis de ambiente para credenciais."
    },
]

for i, dica in enumerate(dicas, 1):
    print(f"\n{'='*60}")
    print(f"Dica {i}: {dica['titulo']}")
    print(f"  RUIM: {dica['ruim']}")
    print(f"  BOM:  {dica['bom'][:100]}..." if len(dica['bom']) > 100 else f"  BOM:  {dica['bom']}")